In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import sqlalchemy as sa

In [29]:
# Evita notación científica al mostrar DataFrames
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
#Creamos la ruta a la bbdd
ruta_db = Path('..') / 'datos'/ 'brutos' / 'ecommerce.db'

#Creamos la conexión a la bbdd
engine = sa.create_engine(f'sqlite:///{ruta_db.resolve()}')

#Probamos la conexión
with engine.connect() as conn:
    print('Conexión exitosa.', conn)

    

Conexión exitosa. <sqlalchemy.engine.base.Connection object at 0x000001FF362692B0>


In [3]:
# Inspeccionamos la bbdd y extraemos todas las tablas.
from sqlalchemy import inspect
inspector = inspect(engine)
tablas = inspector.get_table_names()
print("Tablas en la base de datos: ", tablas)

Tablas en la base de datos:  ['2019-Dec', '2019-Nov', '2019-Oct', '2020-Feb', '2020-Jan']


In [4]:
oct = pd.read_sql('2019-Oct', engine)
nov = pd.read_sql('2019-Nov', engine)
dic = pd.read_sql('2019-Dec', engine)
ene = pd.read_sql('2020-Jan', engine)
feb = pd.read_sql('2020-Feb', engine)

In [5]:
# Comprobamos que todas las tablas tienen la misma estructura y mismo número de columnas.
print(oct.shape, nov.shape, dic.shape, ene.shape, feb.shape)

(407925, 10) (462833, 10) (351304, 10) (443224, 10) (429790, 10)


In [6]:
#Comprobamos que todas las tablas tienen la misma estructura y mismo número de columnas.

print(oct.columns)
print(nov.columns)
print(dic.columns)
print(ene.columns)
print(feb.columns)

Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')


In [7]:
df = pd.concat([oct, nov, dic, ene, feb], axis=0)
df.info()

<class 'pandas.DataFrame'>
Index: 2095076 entries, 0 to 429789
Data columns (total 10 columns):
 #   Column         Dtype  
---  ------         -----  
 0   index          int64  
 1   event_time     str    
 2   event_type     str    
 3   product_id     int64  
 4   category_id    int64  
 5   category_code  str    
 6   brand          str    
 7   price          float64
 8   user_id        int64  
 9   user_session   str    
dtypes: float64(1), int64(4), str(5)
memory usage: 175.8 MB


In [9]:
df.shape

(2095076, 10)

CALIDAD DE DATOS

In [9]:
df.head()

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46 UTC,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55 UTC,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50 UTC,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,122,2019-10-01 00:03:41 UTC,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,124,2019-10-01 00:03:44 UTC,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [10]:
# Como event_time no la ha cogido como datetime, lo convertimos.
df['event_time'] = pd.to_datetime(df['event_time'])
df.head()

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,122,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,124,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 2095076 entries, 0 to 429789
Data columns (total 10 columns):
 #   Column         Dtype              
---  ------         -----              
 0   index          int64              
 1   event_time     datetime64[us, UTC]
 2   event_type     str                
 3   product_id     int64              
 4   category_id    int64              
 5   category_code  str                
 6   brand          str                
 7   price          float64            
 8   user_id        int64              
 9   user_session   str                
dtypes: datetime64[us, UTC](1), float64(1), int64(4), str(4)
memory usage: 175.8 MB


In [12]:
# La variable index no es necesaria. La eliminamos.
df = df.drop(columns=['index'])

In [13]:
# Traducimos los nombres de las columnas a español.
df.columns = ['fecha', 'evento', 'producto', 'categoria', 'categoria_cod',
 'marca', 'precio', 'usuario', 'sesion']
df.head()



,fecha,evento,producto,categoria,categoria_cod,marca,precio,usuario,sesion
0,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [14]:
# ANÁLISIS DE NULOS
# Al hacer info no se veían los nulos. Los saco a parte.
df.isnull().sum()

fecha                  0
evento                 0
producto               0
categoria              0
categoria_cod    2060411
marca             891646
precio                 0
usuario                0
sesion               506
dtype: int64

In [15]:
df.categoria_cod.value_counts()

categoria_cod
appliances.environment.vacuum             14066
stationery.cartrige                        6037
apparel.glove                              5671
furniture.living_room.cabinet              2900
accessories.bag                            2419
furniture.bathroom.bath                    2245
appliances.personal.hair_cutter             460
accessories.cosmetic_bag                    421
appliances.personal.massager                359
appliances.environment.air_conditioner       62
furniture.living_room.chair                  25
Name: count, dtype: int64

Dado que tenemos la categoria_cod tiene casi el 100% de nulos, y además tenemos categoria sin nulos, eliminamos categori_cod, pues sería redundante.
También eliminamos marca porque el 50% son nulos.

In [19]:
df = df.drop(columns=['categoria_cod', 'marca'])

En cuanto a sesión, es una variable muy relevante y no podemos imputarla. Eliminaremos esos 506 registros.

In [20]:
df.dropna(subset = ['sesion'], inplace=True)

In [22]:
# Comprobamos
df.isnull().sum()

fecha        0
evento       0
producto     0
categoria    0
precio       0
usuario      0
sesion       0
dtype: int64

In [24]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 2094570 entries, 0 to 429789
Data columns (total 7 columns):
 #   Column     Non-Null Count    Dtype              
---  ------     --------------    -----              
 0   fecha      2094570 non-null  datetime64[us, UTC]
 1   evento     2094570 non-null  str                
 2   producto   2094570 non-null  int64              
 3   categoria  2094570 non-null  int64              
 4   precio     2094570 non-null  float64            
 5   usuario    2094570 non-null  int64              
 6   sesion     2094570 non-null  str                
dtypes: datetime64[us, UTC](1), float64(1), int64(3), str(2)
memory usage: 127.8 MB


ANÁLISIS VARIABLES NUMÉRICAS

In [30]:
# Realmente la única variable numérica es el precio. El resto son categóricas.
# Apreciamos que la mediana del precio es 4. La media es de 8.4
# Ojo que hay productos con preio negativo. Probablemente son devoluciones.

df.describe()

,producto,categoria,precio,usuario
count,"2,094,570.00","2,094,570.00","2,094,570.00","2,094,570.00"
mean,"5,487,103.56","1,553,112,489,392,098,048.00",8.42,"521,077,545.56"
std,"1,300,923.90","167,907,497,920,480,608.00",19.14,"87,553,855.76"
min,"3,752.00","1,487,580,004,807,082,752.00",-47.62,"4,661,182.00"
25%,"5,724,652.00","1,487,580,005,754,995,456.00",2.05,"480,613,387.00"
50%,"5,811,665.00","1,487,580,008,246,412,288.00",4.00,"553,341,613.00"
75%,"5,858,353.00","1,487,580,013,489,291,520.00",6.86,"578,406,571.00"
max,"5,932,595.00","2,242,903,426,784,559,104.00",327.78,"622,087,993.00"


In [ ]:
print("Precios 0; ", df[df.precio == 0].shape[0])
print("Precios negativos: ", df[df.precio < 0].shape[0])

# Vemos que hay 20533 productos con precio 0.
# Eliminamos esos 20533 productos.

# Vemos que hay 11 productos con precio negativo.
# Probablemente sean devoluciones.
# Eliminamos esos 11 productos.

df = df[df.precio > 0]
df.describe()


Precios 0;  20533
Preios negativos:  11


,producto,categoria,precio,usuario
count,"2,074,026.00","2,074,026.00","2,074,026.00","2,074,026.00"
mean,"5,485,203.09","1,553,192,964,541,272,064.00",8.50,"521,758,909.23"
std,"1,304,219.20","168,128,659,843,838,048.00",19.21,"87,354,480.68"
min,"3,752.00","1,487,580,004,807,082,752.00",0.05,"4,661,182.00"
25%,"5,724,633.00","1,487,580,005,754,995,456.00",2.06,"482,863,572.00"
50%,"5,811,652.00","1,487,580,008,246,412,288.00",4.06,"553,690,395.00"
75%,"5,858,221.00","1,487,580,013,489,291,520.00",6.98,"578,763,499.00"
max,"5,932,585.00","2,242,903,426,784,559,104.00",327.78,"622,087,993.00"


In [ ]:
# ANALISIS VARIABLES CATEGÓRICAS
# Tiene sentido analizar evento, producto y categoria.
# El resto son variables de tipo identificadoras.

# Evento parece correcto.
df.evento.value_counts()



evento
view                961558
cart                574547
remove_from_cart    410357
purchase            127564
Name: count, dtype: int64

In [42]:
# Hay muchísimis productos. No vale la pena hacer un value_counts.
df.producto.nunique()

45327

In [44]:
# Hay muchísimas categorías. No vale la pena hacer un value_counts.
df.categoria.nunique()

508

In [45]:
df.categoria.value_counts(normalize=True)

categoria
1487580007675986893   0.05
1487580005595612013   0.04
1487580005092295511   0.04
1487580005671109489   0.03
1602943681873052386   0.03
                      ... 
1487580013363462335   0.00
1487580008204469224   0.00
1487580011559911545   0.00
1487580009857024046   0.00
2053031020655018687   0.00
Name: proportion, Length: 508, dtype: float64